In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
df_raw_traffic = spark.table("dev_catalog.bronze.raw_traffic")
df_raw_roads = spark.table("dev_catalog.bronze.raw_roads")

In [0]:
df_raw_traffic = df_raw_traffic.drop(col("link_length_km"))
#df_raw_roads = df_raw_roads.drop(col("year") , col("easting"))

In [0]:
df_raw_traffic.display()

In [0]:
df_raw_roads.select("link_length_km").distinct().display()

In [0]:
df_raw_roads = df_raw_roads.withColumn(
    "link_length_km",
    col("link_length_km").try_cast(DoubleType()))

df_raw_traffic = df_raw_traffic.withColumn(
    "count_point_id",
    col("count_point_id").try_cast(IntegerType())
)


In [0]:
df_raw_roads.select("link_length_km").distinct().display()

In [0]:
df_raw_roads.display()

In [0]:
from pyspark.sql.functions import broadcast
df_final = (
   
    df_raw_traffic.join(df_raw_roads.select("count_point_id", "link_length_km"), on="count_point_id", how="left")
    .dropDuplicates(['count_point_id' , "direction_of_travel"])
    .withColumn("recomputed_motor_vehicles" , col('buses_and_coaches') + col('cars_and_taxis') + col('two_wheeled_motor_vehicles') + col('LGVs') + col('HGVs_2_rigid_axle') + col('HGVs_3_rigid_axle') + col('HGVs_4_or_more_rigid_axle') + col('HGVs_3_or_4_articulated_axle') + col('HGVs_5_articulated_axle') + col('HGVs_6_articulated_axle'))
    .withColumn("vehicle_count_consistent" , when(col("all_motor_vehicles") == col("recomputed_motor_vehicles"), True).otherwise(False))
    .withColumn("total_traffic_volume" , col('all_motor_vehicles') + col('pedal_cycles'))
    .withColumn("vehicle_intensity" , when((col('link_length_km').isNull()) | (col('link_length_km') == 0),lit(None))
                .otherwise(col("all_motor_vehicles") / col('link_length_km')))
    .withColumn("is_estimated" , when(col("estimation_method") == "Estimated", True).otherwise(False))
    .withColumn("transformed_time" , current_timestamp())
  
)
    

In [0]:
df_final.display()

In [0]:
df_final.columns

In [0]:
df_final.printSchema

In [0]:
spark.sql("DROP TABLE IF EXISTS dev_catalog.silver.traffic")
# Then write the DataFrame again
df_final.write.mode('overwrite').partitionBy('year').saveAsTable("dev_catalog.silver.traffic")


In [0]:
%sql

  SELECT COUNT(*) FROM dev_catalog.bronze.raw_traffic;

In [0]:
%sql

  SELECT COUNT(*) FROM dev_catalog.silver.traffic;

In [0]:
%sql
DESCRIBE DETAIL dev_catalog.silver.traffic;

In [0]:
%sql
SHOW PARTITIONS dev_catalog.silver.traffic;